# VAW - classification of whole, untrimmed videos (is violence in this video?)
Label = the category folder (Normal -> non-violent, every other category -> violent), so **no start times are needed**. Every video is prepared whole, tracked and described
(geometry text; + optional frame captions), encoded by frozen video models, and classified from video + text + geometry with a nested, grouped cross-validation.

Inputs: the shared `violence` folder. Outputs: `VAW_results/classfull/`. Everything is resumable: after a disconnect run all cells again.
Set `MAX_CLIPS` (for example 20) for a quick trial before the full run. Switch **Layer B** (frame captions by a video-language model) on or off with `USE_CAPTIONS`.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/KRITHIKSAI16/Violence-Against-Women.git"
if not os.path.isdir('/content/VAW'):
    !git clone -q {url} /content/VAW
else:
    !git -C /content/VAW pull -q {url} main
%cd /content/VAW
!pip install -q -r colabclassfull/requirements.txt
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (switch the runtime to a T4 GPU: Runtime > Change runtime type)')
!python -m pytest -q tests/test_class_text_geom.py tests/test_class_metrics_cv.py tests/test_class_pipeline.py

## Settings

In [ ]:
OUT = '/content/drive/Shareddrives/VAW/VAW_results/classfull'
DATA_ROOT = '/content/drive/Shareddrives/VAW/violence'
CURATED = '/content/drive/Shareddrives/VAW/VAW_results/curated_previolence_v2'   # optional: its perception_choice.json is reused; set None to skip
USE_CAPTIONS = True      # Layer B on (default). False = geometry text only
MAX_CLIPS = None         # e.g. 20 for a quick trial run
flag = '' if USE_CAPTIONS else '--no-captions'
common = f"--out-root {OUT} {flag}"
if CURATED and os.path.isdir(CURATED):
    common += f" --curated-root {CURATED}"
trial = f" --max-clips {MAX_CLIPS}" if MAX_CLIPS else ''
assert os.path.isdir(DATA_ROOT), 'video folder not found: fix DATA_ROOT'

## 1. Prepare: list the videos, re-encode each WHOLE (30 fps, 640 px), write the manifest

In [ ]:
!python -m src.classfull.run --stage prepare {common} --data-root '{DATA_ROOT}'{trial}

## 2. Shots and tracking (longest step; GPU)

In [ ]:
!python -m src.classfull.run --stage track {common}

## 3. Camera, distances, scene layout and depth, story (GPU for scene/depth)

In [ ]:
!python -m src.classfull.run --stage context {common}

## 4. Analysis of every clip: the pair facts the text is built from

In [ ]:
!python -m src.classfull.run --stage analyze {common}

## 5. Layer B: describe frames of every clip (GPU; skipped when USE_CAPTIONS is False; resumable)

In [ ]:
!python -m src.classfull.run --stage captions {common}

## 6. Video features (frozen X-CLIP / V-JEPA 2 / InternVideo2; GPU)

In [ ]:
!python -m src.classfull.run --stage features {common}

## 7. Train, cross-validate, report

In [ ]:
!python -m src.classfull.run --stage train {common}

## 8. Look at it here

In [ ]:
from IPython.display import Image, display
print(open(OUT + '/report/report.md', encoding='utf-8').read())
if os.path.exists(OUT + '/report/roc.png'):
    display(Image(OUT + '/report/roc.png'))